# Veritas AI: Multimodal Content Verification — Google Colab Training Pipeline

**Author:** University NLP Capstone Team  
**Task:** Multimodal Fake News Classification & Selective Uncertainty (REAL / FAKE / UNCERTAIN)  
**Dataset:** Fakeddit Multimodal Benchmark (Nakamura et al., 2020)  
**Hardware Target:** Google Colab GPU (T4, V100, or A100)

---
### Pipeline Overview:
1. **Environment Setup & Drive Persistence**: GPU verification and Google Drive checkpoint mounting.
2. **Dataset Acquisition & Pillow Validation**: Fakeddit multimodal pair filtering and stratified 70/15/15 splitting.
3. **Unimodal Baselines**: DistilBERT text classification and ResNet50 vision classification.
4. **Out-of-Fold (OOF) Logit Generation**: Leakage-free training for late fusion meta-classifier.
5. **Multimodal Fusion Architectures**: Feature-Level Fusion (Intermediate Concatenation) vs Decision-Level Fusion.
6. **Temperature Scaling (NLL Optimization)**: Validation-calibrated confidence refinement.
7. **Selective Classification (Uncertainty $\tau^*$)**: Empirical validation threshold search for $\ge 90\%$ selective accuracy.
8. **Untouched Test Evaluation**: Benchmark testing, ECE, Brier score, and JSON artifact export.

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("WARNING: Running on CPU. Recommend switching to GPU in Runtime -> Change runtime type.")

In [ ]:
# Step 2: Mount Google Drive to persist checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/Veritas_AI_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
# Step 3: Install dependencies
!pip install -q transformers>=4.41.0 shap>=0.45.0 scikit-learn>=1.4.0 scipy>=1.12.0 pillow>=10.2.0

## 2. Dataset Ingestion & Preprocessing
Acquires verified Fakeddit paired multimodal samples (balanced REAL / FAKE) and performs stratified splitting.

In [ ]:
# Set seeds for scientific reproducibility
import random, numpy as np, torch
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
print("Random seed locked to 42.")

## 3. Unimodal & Multimodal PyTorch Architectures
- **DistilBERT**: Text classification head (`Linear(768, 2)`)
- **ResNet50**: Image classification head (`Linear(2048, 256) -> ReLU -> Linear(256, 2)`)
- **FeatureFusionNet**: Projected joint representation $\mathbb{R}^{256} + \mathbb{R}^{256} \to \mathbb{R}^{512}$ with MLP classifier.

In [ ]:
import torch.nn as nn
from torchvision import models
from transformers import DistilBertModel

class DistilBertBaseline(nn.Module):
    def __init__(self, pretrained_model='distilbert-base-uncased', num_classes=2):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(pretrained_model)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_token))
        return logits, cls_token

class ResNet50Baseline(nn.Module):
    def __init__(self, num_classes=2, freeze_early=True):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        if freeze_early:
            for param in list(backbone.parameters())[:-30]:
                param.requires_grad = False
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # Output: (B, 2048, 1, 1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, images):
        feat = self.backbone(images).squeeze(-1).squeeze(-1)
        logits = self.classifier(feat)
        return logits, feat

class FeatureFusionNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.text_bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        rnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for param in list(rnet.parameters())[:-30]:
            param.requires_grad = False
        self.img_backbone = nn.Sequential(*list(rnet.children())[:-1])

        self.proj_text = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        self.proj_image = nn.Sequential(
            nn.Linear(2048, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, images):
        text_out = self.text_bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = text_out.last_hidden_state[:, 0, :]
        img_emb = self.img_backbone(images).squeeze(-1).squeeze(-1)

        e_t = self.proj_text(cls_emb)
        e_i = self.proj_image(img_emb)
        fused = torch.cat([e_t, e_i], dim=-1)
        logits = self.classifier(fused)
        return logits, e_t, e_i

print("Model architectures successfully initialized.")

## 4. Post-Hoc Temperature Scaling (NLL Optimization)
Optimizes scalar parameter $T > 0$ on validation set logits strictly by minimizing Negative Log-Likelihood.

In [ ]:
from scipy.optimize import minimize

def optimize_temperature(val_logits: np.ndarray, val_labels: np.ndarray) -> float:
    """Optimize T > 0 on validation logits using Negative Log-Likelihood."""
    def nll_objective(t_arr):
        t = t_arr[0]
        scaled_logits = val_logits / t
        exp_logits = np.exp(scaled_logits - np.max(scaled_logits, axis=-1, keepdims=True))
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        # NLL
        correct_probs = probs[np.arange(len(val_labels)), val_labels]
        nll = -np.mean(np.log(np.clip(correct_probs, 1e-12, 1.0)))
        return nll

    res = minimize(nll_objective, x0=[1.0], bounds=[(0.1, 10.0)], method='L-BFGS-B')
    opt_t = float(res.x[0])
    print(f"Optimized Temperature T*: {opt_t:.4f} (Validation NLL: {res.fun:.4f})")
    return opt_t

def compute_brier_score(probs: np.ndarray, labels: np.ndarray) -> float:
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(labels)), labels] = 1.0
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=-1)))

print("Calibration functions defined.")

## 5. Selective Classification & Empirical Threshold ($\\tau^*$)
Sweeps confidence thresholds on the validation set to target $\ge 90\%$ selective accuracy, reporting coverage.

In [ ]:
def sweep_rejection_threshold(calibrated_probs: np.ndarray, labels: np.ndarray, target_acc: float = 0.90):
    """Find tau* on validation set targeting selective accuracy >= target_acc."""
    preds = np.argmax(calibrated_probs, axis=-1)
    confidences = np.max(calibrated_probs, axis=-1)

    best_tau = 0.50
    best_cov = 1.0
    best_acc = np.mean(preds == labels)

    thresholds = np.linspace(0.50, 0.99, 50)
    for tau in thresholds:
        accepted = confidences >= tau
        if np.sum(accepted) < 20: # Require at least 20 samples
            continue
        sel_acc = np.mean(preds[accepted] == labels[accepted])
        cov = np.mean(accepted)
        if sel_acc >= target_acc:
            best_tau = float(tau)
            best_acc = float(sel_acc)
            best_cov = float(cov)
            break

    print(f"Empirical Threshold tau*: {best_tau:.4f}")
    print(f"Validation Selective Accuracy: {best_acc*100:.2f}%")
    print(f"Validation Coverage: {best_cov*100:.2f}%")
    return best_tau, best_acc, best_cov

print("Selective threshold logic defined.")